In [14]:
from pathlib import Path
import pandas as pd
import re
import html
import nltk
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to C:\Users\RAHUL
[nltk_data]     MANGUTKAR\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\RAHUL
[nltk_data]     MANGUTKAR\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_path = RAW_DATA_DIR / "banking77_train.csv"
test_path = RAW_DATA_DIR / "banking77_test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTraining columns:", train_df.columns.tolist())

assert {"text", "category"}.issubset(train_df.columns)
assert {"text", "category"}.issubset(test_df.columns)

processed_df = train_df.copy()

print("\nOriginal training data loaded successfully.")

Training shape: (10003, 2)
Test shape: (3080, 2)

Training columns: ['text', 'category']

Original training data loaded successfully.


### Normalization function

In [3]:
def normalize_text(text):
    text = html.unescape(str(text))
    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    return text

In [4]:
processed_df["text_normalized"] = processed_df["text"].apply(normalize_text)

processed_df[["text", "text_normalized", "category"]].head()

,text,text_normalized,category
0,I am still waiting on my card?,i am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,what can i do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,i have been waiting over a week. is the card s...,card_arrival
3,Can I track my card while it is in the process...,can i track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...","how do i know if i will get my card, or if it ...",card_arrival


### Handle URLs

In [5]:
def handle_urls(text):
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " URL ",
        text
    )

    return re.sub(r"\s+", " ", text).strip()

In [6]:
processed_df["text_no_urls"] = processed_df["text_normalized"].apply(handle_urls)

processed_df[["text_normalized", "text_no_urls"]].head()

,text_normalized,text_no_urls
0,i am still waiting on my card?,i am still waiting on my card?
1,what can i do if my card still hasn't arrived ...,what can i do if my card still hasn't arrived ...
2,i have been waiting over a week. is the card s...,i have been waiting over a week. is the card s...
3,can i track my card while it is in the process...,can i track my card while it is in the process...
4,"how do i know if i will get my card, or if it ...","how do i know if i will get my card, or if it ..."


### Handle repeated punctuation

In [7]:
def handle_punctuation(text):
    text = re.sub(r"([!?.,])\1+", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [8]:
processed_df["text_clean_punctuation"] = processed_df["text_no_urls"].apply(handle_punctuation)

### Handle contractions

In [9]:
contractions = {
    "can't": "can not",
    "cannot": "can not",
    "couldn't": "could not",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hasn't": "has not",
    "haven't": "have not",
    "isn't": "is not",
    "mustn't": "must not",
    "shouldn't": "should not",
    "wasn't": "was not",
    "weren't": "were not",
    "won't": "will not",
    "wouldn't": "would not",
    "it's": "it is",
    "that's": "that is",
    "there's": "there is",
    "I've": "i have",
    "I'm": "i am",
    "you're": "you are",
    "we're": "we are",
    "they're": "they are",
    "I've": "i have"
}

In [10]:
def expand_contractions(text):
    text = text.lower()

    for contraction, expanded in contractions.items():
        pattern = r"\b" + re.escape(contraction) + r"\b"
        text = re.sub(pattern, expanded, text)

    return re.sub(r"\s+", " ", text).strip()

In [11]:
processed_df["text_expanded"] = processed_df["text_clean_punctuation"].apply(expand_contractions)

### Tokenization

In [12]:
def tokenize_text(text):
    return re.findall(r"\b\w+\b", text.lower())

In [13]:
processed_df["tokens"] = processed_df["text_expanded"].apply(tokenize_text)

processed_df[["text_expanded", "tokens"]].head()

,text_expanded,tokens
0,i am still waiting on my card?,"[i, am, still, waiting, on, my, card]"
1,what can i do if my card still has not arrived...,"[what, can, i, do, if, my, card, still, has, n..."
2,i have been waiting over a week. is the card s...,"[i, have, been, waiting, over, a, week, is, th..."
3,can i track my card while it is in the process...,"[can, i, track, my, card, while, it, is, in, t..."
4,"how do i know if i will get my card, or if it ...","[how, do, i, know, if, i, will, get, my, card,..."


### Stop-word removal

In [15]:
negation_words = {"no", "not", "nor", "never"}

stop_words = set(ENGLISH_STOP_WORDS) - negation_words

In [16]:
def remove_stop_words(tokens):
    return [
        token
        for token in tokens
        if token not in stop_words]

In [17]:
processed_df["tokens_no_stopwords"] = processed_df["tokens"].apply(remove_stop_words)

### Stemming

In [18]:
stemmer = PorterStemmer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

In [19]:
processed_df["tokens_stemmed"] = processed_df["tokens"].apply(stem_tokens)

### Lemmatization

In [20]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [
        lemmatizer.lemmatize(token)
        for token in tokens]

In [21]:
processed_df["tokens_lemmatized"] = processed_df["tokens"].apply(lemmatize_tokens)

### Compare all preprocessing experiments

In [23]:
comparison_columns = [
    "text",
    "text_normalized",
    "text_no_urls",
    "text_clean_punctuation",
    "text_expanded",
    "tokens",
    "tokens_no_stopwords",
    "tokens_stemmed",
    "tokens_lemmatized",
    "category"
]

display(processed_df[comparison_columns].sample(5))

,text,text_normalized,text_no_urls,text_clean_punctuation,text_expanded,tokens,tokens_no_stopwords,tokens_stemmed,tokens_lemmatized,category
242,"I recently reported my card as missing/lost, a...","i recently reported my card as missing/lost, a...","i recently reported my card as missing/lost, a...","i recently reported my card as missing/lost, a...","i recently reported my card as missing/lost, a...","[i, recently, reported, my, card, as, missing,...","[recently, reported, card, missing, lost, just...","[i, recent, report, my, card, as, miss, lost, ...","[i, recently, reported, my, card, a, missing, ...",card_linking
682,There is an unexplained $1 charge on my statem...,there is an unexplained $1 charge on my statem...,there is an unexplained $1 charge on my statem...,there is an unexplained $1 charge on my statem...,there is an unexplained $1 charge on my statem...,"[there, is, an, unexplained, 1, charge, on, my...","[unexplained, 1, charge, statement, explain]","[there, is, an, unexplain, 1, charg, on, my, s...","[there, is, an, unexplained, 1, charge, on, my...",extra_charge_on_statement
2492,I asked for cash but I did not receive all the...,i asked for cash but i did not receive all the...,i asked for cash but i did not receive all the...,i asked for cash but i did not receive all the...,i asked for cash but i did not receive all the...,"[i, asked, for, cash, but, i, did, not, receiv...","[asked, cash, did, not, receive, funds]","[i, ask, for, cash, but, i, did, not, receiv, ...","[i, asked, for, cash, but, i, did, not, receiv...",wrong_amount_of_cash_received
2828,Why is the transaction to my friend still pend...,why is the transaction to my friend still pend...,why is the transaction to my friend still pend...,why is the transaction to my friend still pend...,why is the transaction to my friend still pend...,"[why, is, the, transaction, to, my, friend, st...","[transaction, friend, pending]","[whi, is, the, transact, to, my, friend, still...","[why, is, the, transaction, to, my, friend, st...",transfer_not_received_by_recipient
173,"I would like to re-activate my card, it was pr...","i would like to re-activate my card, it was pr...","i would like to re-activate my card, it was pr...","i would like to re-activate my card, it was pr...","i would like to re-activate my card, it was pr...","[i, would, like, to, re, activate, my, card, i...","[like, activate, card, previously, reported, l...","[i, would, like, to, re, activ, my, card, it, ...","[i, would, like, to, re, activate, my, card, i...",card_linking


In [24]:
processed_df["text_no_stopwords"] = processed_df["tokens_no_stopwords"].apply(lambda tokens: " ".join(tokens))

processed_df["text_stemmed"] = processed_df["tokens_stemmed"].apply(lambda tokens: " ".join(tokens))

processed_df["text_lemmatized"] = processed_df["tokens_lemmatized"].apply(lambda tokens: " ".join(tokens))

### Validate the processed data

In [25]:
print("Original rows:", len(train_df))
print("Processed rows:", len(processed_df))

print("\nMissing values in original text:")
print(train_df["text"].isna().sum())

print("\nMissing values in normalized text:")
print(processed_df["text_normalized"].isna().sum())

print("\nOriginal category count:", train_df["category"].nunique())
print("Processed category count:", processed_df["category"].nunique())

print(
    "\nAre the original labels unchanged?",
    train_df["category"].equals(processed_df["category"])
)

assert len(train_df) == len(processed_df)
assert train_df["category"].equals(processed_df["category"])

Original rows: 10003
Processed rows: 10003

Missing values in original text:
0

Missing values in normalized text:
0

Original category count: 77
Processed category count: 77

Are the original labels unchanged? True


### Save the experimental dataset

In [27]:
output_path = (
    PROCESSED_DATA_DIR/ "phase3_train_preprocessing_experiments.csv")

processed_df.to_csv(output_path, index=False)

print("Saved preprocessing experiments to:")
print(output_path)

Saved preprocessing experiments to:
c:\Users\RAHUL MANGUTKAR\banking-customer-intelligence\data\processed\phase3_train_preprocessing_experiments.csv


### Observations

Text normalization provides a consistent representation of messages with different capitalization and spacing.

URLs can be replaced with a placeholder when their exact contents are not needed for intent classification.

Punctuation should be handled carefully because aggressive removal may discard useful information.

Contraction handling is important because expressions such as “didn't” and “don't” carry meaning, particularly in reports of unauthorized banking transactions.

Negation words such as “not,” “no,” and “never” should not be blindly removed.

Stemming and lemmatization produce different word representations and should be evaluated rather than assumed to improve model performance.

The preprocessing output was saved separately, and the original text and category labels were retained for comparison.